In [1]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import torchvision.transforms as transforms
import torch.utils.data as data

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Upload dataset, Remove nulls

In [3]:
df = pd.read_pickle('drive/MyDrive/dataset22462_augmented.pkl')
# df = df.dropna().drop(['NAME','Scale','Activity','Category','ID'], axis=1)
#convert the columns upper neck_X upper neck_Y  head top_X  head top_Y to int64
df = df.astype({'upper neck_X': 'int64', 'upper neck_Y': 'int64', 'head top_X': 'int64', 'head top_Y': 'int64'})

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22346 entries, 0 to 22345
Data columns (total 33 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   r ankle_X     22346 non-null  float64
 1   r ankle_Y     22346 non-null  float64
 2   r knee_X      22346 non-null  float64
 3   r knee_Y      22346 non-null  float64
 4   r hip_X       22346 non-null  float64
 5   r hip_Y       22346 non-null  float64
 6   l hip_X       22346 non-null  float64
 7   l hip_Y       22346 non-null  float64
 8   l knee_X      22346 non-null  float64
 9   l knee_Y      22346 non-null  float64
 10  l ankle_X     22346 non-null  float64
 11  l ankle_Y     22346 non-null  float64
 12  pelvis_X      22346 non-null  float64
 13  pelvis_Y      22346 non-null  float64
 14  thorax_X      22346 non-null  float64
 15  thorax_Y      22346 non-null  float64
 16  upper neck_X  22346 non-null  int64  
 17  upper neck_Y  22346 non-null  int64  
 18  head top_X    22346 non-nu

## Contruct the Dataset Object

In [5]:
keypoint_columns = [col for col in df.columns if col != 'image'] #todas as coluunas
keypoints = df[keypoint_columns].values.astype(np.float32)

keypoints = keypoints / 224.0 #normaliza os pontos para 0-1

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

class PoseDataset(data.Dataset):
    def __init__(self, df):
        self.df = df
        self.images = df['image'].tolist()
        self.keypoints = keypoints
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_array = self.images[idx]
        img_tensor = transform(img_array) #converter para pythorch
        kp_tensor = torch.tensor(self.keypoints[idx])
        return img_tensor, kp_tensor

## Spliting for test and train

In [6]:
dataset = PoseDataset(df)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) -train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = data.DataLoader(val_dataset, batch_size=32, shuffle=False)

## Defining the model

In [25]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # Skip connection (identity mapping)
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)  # Adding the residual connection
        out = self.relu(out)
        return out

class PoseNet(nn.Module):
    def __init__(self):
        super(PoseNet, self).__init__()
        # Using residual blocks for feature extraction
        self.conv = nn.Sequential(
            ResidualBlock(3, 32),    # Initial Conv + Residual Block
            nn.MaxPool2d(2, 2),      # 112x112

            ResidualBlock(32, 64),   # Residual Block
            nn.MaxPool2d(2, 2),      # 56x56

            ResidualBlock(64, 128),  # Residual Block
            nn.MaxPool2d(2, 2),      # 28x28

            ResidualBlock(128, 256),  # Residual Block
            nn.MaxPool2d(2, 2),      # 28x28

            ResidualBlock(256, 512), # Residual Block
            nn.AdaptiveAvgPool2d((14, 14))  # 7x7 output
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, 32),  # Assuming 16 keypoints, each with x, y = 32 values
            nn.Sigmoid()  # Output keypoint coordinates between [0,1]
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [ ]:


# class PoseNet(nn.Module):
#     def __init__(self):
#         super(PoseNet, self).__init__()
#         self.conv = nn.Sequential(
#             nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
#             nn.ReLU(),
#             nn.MaxPool2d(2, 2),  # 112x112
#             nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
#             nn.ReLU(),
#             nn.MaxPool2d(2, 2),  # 56x56
#             nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
#             nn.ReLU(),
#             nn.MaxPool2d(2, 2),  # 28x28
#         )
#         self.fc = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(128 * 28 * 28, 512),
#             nn.ReLU(),
#             nn.Linear(512, 32)
#         )

#     def forward(self, x):
#         x = self.conv(x)
#         x = self.fc(x)
#         return x

## Training the model

In [26]:
model = PoseNet().to("cuda")
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.00125, weight_decay=2e-5)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MSEcriterion = nn.MSELoss()
epochs = 20
for epoch in range(epochs):
    model.train()
    train_smooth_loss, train_mse_loss = 0.0, 0.0

    for images, keypoints in train_loader:
        if device.type == 'cuda':
            images, keypoints = images.to("cuda"), keypoints.to("cuda")

        optimizer.zero_grad()
        outputs = model(images)

        smooth_loss = criterion(outputs, keypoints)
        mse_loss = MSEcriterion(outputs, keypoints)  # Just for monitoring

        smooth_loss.backward()
        optimizer.step()

        train_smooth_loss += smooth_loss.item()
        train_mse_loss += mse_loss.item()

    # scheduler.step(smooth_loss)
    avg_train_smooth_loss = train_smooth_loss / len(train_loader)
    avg_train_mse_loss = train_mse_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}], Train Loss (SmoothL1): {avg_train_smooth_loss:.4f}, Train Loss (MSE): {avg_train_mse_loss:.4f}")

    # Validation
    if(epoch+1) % 3 == 0:
        model.eval()
        val_smooth_loss, val_mse_loss = 0.0, 0.0
        with torch.no_grad():
            for images, keypoints in val_loader:
                if device.type == 'cuda':
                    images, keypoints = images.to("cuda"), keypoints.to("cuda")
                    outputs = model(images)
                smooth_loss = criterion(outputs, keypoints)
                mse_loss = MSEcriterion(outputs, keypoints)  # Just for monitoring

                val_smooth_loss += smooth_loss.item()
                val_mse_loss += mse_loss.item()

        avg_val_smooth_loss = val_smooth_loss / len(val_loader)
        avg_val_mse_loss = val_mse_loss / len(val_loader)
        print(f"Val Loss (SmoothL1): {avg_val_smooth_loss:.4f}, Val Loss (MSE): {avg_val_mse_loss:.4f}")

Epoch [1/20], Train Loss (SmoothL1): 0.0205, Train Loss (MSE): 0.0426


## Training logs
lr=0.01; epoch: 10; Loss: 0.0014; Val Loss: 0.0441

In [ ]:
torch.save(model.state_dict(), "posev00178.pth")
